In [1]:
import json
import numpy as np
import pandas as pd
from scipy.optimize import minimize

In [3]:
# 1. CARGAR PROBABILIDADES DEL HMM, COSTOS Y PARÁMETROS DE COLAS
# ==============================================================================
with open("../datos/resultados_hmm.json", "r", encoding="utf-8") as f:
    datos_hmm = json.load(f)

A = np.array(datos_hmm["matriz_transicion"])
estados = ["E0 (Reconnaissance)", "E1 (Initial Access)", "E2 (Exfiltration)", "E3 (Pivote)"]

# Calculamos la distribución estacionaria (pi) de la Cadena de Markov (pi * A = pi)
# Representa la proporción de tiempo/flujos en cada estado en régimen permanente
eigvals, eigvecs = np.linalg.eig(A.T)
pi_estacionaria = np.real(eigvecs[:, np.isclose(eigvals, 1.0)][:, 0])
pi_estacionaria = pi_estacionaria / pi_estacionaria.sum()

# Costo de impacto base por estado (de Teoría de Decisiones)
# E0: 10 (riesgo de escalada), E1: 30, E2: 100 (exfiltración crítica), E3: 40 (pivote)
C_impacto = np.array([10.0, 30.0, 100.0, 40.0])

# Efectividad de mitigación (alpha_i) por cada unidad de capacidad (flujos/seg) asignada
alpha = np.array([0.8, 0.6, 0.5, 0.6])

# Parámetros de capacidad de hardware extraídos de Teoría de Colas
B_capacidad_max = 6.8728  # Capacidad máxima total del servidor IPS (mu_base flujos/seg)
K_latencia = 5.0          # Factor de penalización económica por congestión de cola M/M/1

print("--- Distribución Estacionaria (π) del Sistema ---")
for est, p in zip(estados, pi_estacionaria):
    print(f"  {est:22s}: {p*100:6.2f}%")

--- Distribución Estacionaria (π) del Sistema ---
  E0 (Reconnaissance)   :  77.39%
  E1 (Initial Access)   :  11.66%
  E2 (Exfiltration)     :   2.15%
  E3 (Pivote)           :   8.80%


In [5]:
# 2. FORMULACIÓN DEL PROBLEMA DE PROGRAMACIÓN NO LINEAL (PNL)
# ==============================================================================
# Variables de decisión: x = [x0, x1, x2, x3] (capacidad en flujos/seg asignada a cada estado)
# Función Objetivo: Minimizar Riesgo Residual Exponencial + Costo Hiperbólico de Cola M/M/1
def funcion_objetivo(x):
    riesgo_residual = np.sum(C_impacto * pi_estacionaria * np.exp(-alpha * x))
    holgura_servidor = B_capacidad_max - np.sum(x)
    costo_congestion = K_latencia / holgura_servidor
    return riesgo_residual + costo_congestion

# Restricción de desigualdad: sum(x) <= B_capacidad_max - margen_seguridad
# En scipy (SLSQP), las restricciones se expresan como g(x) >= 0
margen_seguridad = 0.05
restricciones = [
    {"type": "ineq", "fun": lambda x: (B_capacidad_max - margen_seguridad) - np.sum(x)}
]

# Cotas de no negatividad: x_i >= 0
cotas = [(0.0, B_capacidad_max) for _ in range(4)]

# Punto inicial factible (reparto equitativo conservador)
x0 = np.array([1.0, 1.0, 1.0, 1.0])

# Resolución mediante SLSQP (Sequential Least Squares Programming)
resultado = minimize(
    funcion_objetivo,
    x0,
    method="SLSQP",
    bounds=cotas,
    constraints=restricciones,
    options={"ftol": 1e-9, "disp": False}
)

In [6]:
# 3. RESULTADOS DE LA ASIGNACIÓN ÓPTIMA DE RECURSOS
# ==============================================================================
x_opt = resultado.x
capacidad_total_usada = np.sum(x_opt)
utilizacion_optima = (capacidad_total_usada / B_capacidad_max) * 100
latencia_resultante = 1.0 / (B_capacidad_max - capacidad_total_usada)

df_pnl = pd.DataFrame({
    "Estado HMM": estados,
    "Prob. Estacionaria (%)": (pi_estacionaria * 100).round(2),
    "Costo Impacto (Ci)": C_impacto,
    "Capacidad Asignada x* (flujos/s)": x_opt.round(4),
    "% del Servidor Asignado": ((x_opt / B_capacidad_max) * 100).round(2)
})

print("\n========================================================")
print(f"ÓPTIMO GLOBAL ALCANZADO (Costo Total Mínimo): {resultado.fun:.4f}")
print("========================================================")
print(df_pnl.to_string(index=False))

print("\n--- Métricas Operativas del Servidor IPS en el Óptimo ---")
print(f"Capacidad Total Utilizada : {capacidad_total_usada:.4f} / {B_capacidad_max:.4f} flujos/seg")
print(f"Utilización Óptima (ρ*)   : {utilizacion_optima:.2f}%")
print(f"Latencia Total W* (M/M/1) : {latencia_resultante:.4f} segundos")


ÓPTIMO GLOBAL ALCANZADO (Costo Total Mínimo): 9.0890
         Estado HMM  Prob. Estacionaria (%)  Costo Impacto (Ci)  Capacidad Asignada x* (flujos/s)  % del Servidor Asignado
E0 (Reconnaissance)                   77.39                10.0                            2.2359                    32.53
E1 (Initial Access)                   11.66                30.0                            1.1784                    17.15
  E2 (Exfiltration)                    2.15               100.0                            0.0715                     1.04
        E3 (Pivote)                    8.80                40.0                            1.1891                    17.30

--- Métricas Operativas del Servidor IPS en el Óptimo ---
Capacidad Total Utilizada : 4.6749 / 6.8728 flujos/seg
Utilización Óptima (ρ*)   : 68.02%
Latencia Total W* (M/M/1) : 0.4550 segundos
